# Load Sentiment CSV into GHTorrent MySQL (Notebook 1)

This notebook shows how to load the GitHub Gold Standard sentiment CSV into a MySQL database that already has the GHTorrent 2004 dump. It also includes quick checks to make sure the data loaded correctly.

### Planned Output
By the end of this notebook, you should have:
1. A `comment_sentiment` table in MySQL
2. All rows from `comment_sentiment.csv` loaded
3. Query results that confirm row counts and valid joins to GHTorrent project data

### Step 1: Get the data ready

1. Download the [GitHub Gold Standard dataset](https://figshare.com/articles/dataset/A_gold_standard_for_polarity_of_emotions_of_software_developers_in_GitHub/11604597?file=21001260).
2. Rename the file to `comment_sentiment.csv`.
3. Download the [GHTorrent 2004 MySQL Database Dump](https://web.archive.org/web/20150206005357/http://ghtorrent.org/msr14.html) and make sure it is already loaded in your MySQL database (example: `github`).
4. Make sure MySQL can read your CSV file path (e.g., `~/Desktop/github/sentiment_github_dataset/comment_sentiment.csv`)

Optional reference: [GHTorrent schema diagram](https://web.archive.org/web/20150206005412/http://ghtorrent.org/relational.html).

### Step 2: Create the table, load the CSV, and run the original join queries

Use these copy-ready blocks one at a time.

Start MySQL with local file loading turned on. Run on bash:

```bash
mysql --local-infile=1 -u root -p
```
---

Select your database (e.g., `github`):

```sql
USE github;
```
---

Drop old table if it exists (safe to re-run):

```sql
DROP TABLE IF EXISTS comment_sentiment;
```

---

Create table:

```sql
CREATE TABLE comment_sentiment (
  ID INT NULL,
  Polarity VARCHAR(256) NULL,
  Text TEXT NULL
);
```

---

Load CSV (replace with your absolute path if needed):

```sql
LOAD DATA LOCAL INFILE 'comment_sentiment.csv'
INTO TABLE comment_sentiment
FIELDS TERMINATED BY ';'
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(ID, Polarity, Text);
```

---

Query 1 — show joined sentiment + commit + project rows (sample view):

```sql
-- Returns joined rows from sentiment comments to commit/project data
SELECT * FROM comment_sentiment s
INNER JOIN commit_comments cc ON s.ID = cc.comment_id
INNER JOIN commits c ON c.id = cc.commit_id
INNER JOIN projects p ON c.project_id = p.id;
```

---

Query 2 — count sentiment-linked comments by project name:

```sql
-- Aggregates joined rows by project name and sorts by largest counts
SELECT name, count(name) as count FROM comment_sentiment s
INNER JOIN commit_comments cc ON s.ID = cc.comment_id
INNER JOIN commits c ON c.id = cc.commit_id
INNER JOIN projects p ON c.project_id = p.id
GROUP BY name
ORDER BY count desc;
```

### Step 3: Run validation checks

Use these checks to confirm the load worked correctly.

Check 1 — total rows (expected: 7122):

```sql
SELECT COUNT(*) AS total_rows FROM comment_sentiment;
```

---

Check 2 — distinct comment IDs (expected: 7122):

```sql
SELECT COUNT(DISTINCT ID) AS distinct_comment_ids FROM comment_sentiment;
```

### Optional troubleshooting

If `LOAD DATA LOCAL INFILE` fails or the row count is too low:

1. Check the row count. If it is below 7,122 comments, try the fixes below:

```sql
SELECT COUNT(*) AS total_rows FROM comment_sentiment;
```

2. Try these fixes:
- Use an absolute file path in `LOAD DATA LOCAL INFILE`
- Make sure `--local-infile=1` is enabled
- Make sure the file format matches your settings (`;` delimiter and quoted text)

3. If needed, use the following Python CSV loader script (([import_csv_to_mysql.py](https://github.com/user-attachments/files/25094159/import_csv_to_mysql.py))), then run the same checks again. This option uses Python's CSV parser and requires the installation of `mysql-connector-python`.

### When to move to Notebook 2

Move to Notebook 2 only after `total_rows = 7122` and join results are greater than zero.